---

# Identificación de Clases

---

## Extracción de Clases Candidatas

Las clases se identifican a partir de sustantivos en los casos de uso y su contexto. Aquí el análisis:

| Fuente           | Sustantivos Identificados                          | Clase Candidata o Atributo?             |
|------------------|---------------------------------------------------|-----------------------------------------|
| Caso de Uso 1    | Usuario, Carnet, Contraseña, Rol, Sistema         | Usuario (clase), Rol (atributo)         |
| Caso de Uso 2    | Oferta Académica, Asignatura, Carrera, Sede, Ciclo| OfertaAcademica, Asignatura, Carrera     |
| Caso de Uso 3    | Matrícula, Cupos, Constancia, Deudas (futuro)     | Matricula, Asignatura                    |
| Caso de Uso 4    | Período de modificación                           | Periodo                                  |
| Caso de Uso 5    | Períodos, Fechas                                  | Periodo (ya identificada)               |
| Caso de Uso 6    | Cupos, Sección, Jefe de Carrera                   | JefeCarrera, Asignatura                  |
| Caso de Uso 7    | Reporte, Criterios, Formato (PDF/Excel)           | Reporte                                  |



**Clases Candidatas Finales**:

Usuario, Estudiante, Administrativo, JefeCarrera, Asignatura, Matricula, OfertaAcademica, Periodo, Reporte.

## Clases Abstractas vs. Finales

**Clases Abstractas**

Son clases que <u>no se instancian directamente</u> y sirven como base para otras. Se identifican cuando hay una <u>generalización</u> de comportamientos o atributos:

Usuario:
- Razón: Es una generalización de Estudiante, Administrativo, y JefeCarrera.
- Atributos/Métodos Comunes: id, contraseña, rol, autenticar().
- UML: Se marca como abstract en el diagrama.

**Clases Finales**

Son clases que <u>no pueden heredarse</u> (no tienen subclases). Se identifican cuando no hay especialización futura:

- Asignatura: Representa una entidad concreta sin variantes (ej: no hay "AsignaturaElectiva" o "AsignaturaObligatoria" en los casos de uso).
- Periodo: Solo almacena fechas de apertura/cierre, sin comportamientos complejos.
- Reporte: Su función es generar datos, sin necesidad de subclases (PDF, Excel son formatos, no clases).

**Justificación de Exclusiones**

Algunos sustantivos no se convierten en clases:

- Carrera, Sede, Ciclo: Son atributos de Estudiante o OfertaAcademica (ej: carrera: String).
- Deudas: Mencionado como "fase futura", no es relevante para el modelo actual.
- Constancia: Es un documento generado, no una entidad con comportamiento.

**Principios UML Aplicados**
 
- Abstracción: Usuario abstrae lo común entre estudiantes, administrativos y jefes.
- Encapsulamiento: Atributos como contraseña son privados (-), métodos como autenticar() son públicos (+).
- Cohesión: Cada clase tiene responsabilidades claras (ej: Matricula gestiona asignaturas, no cálculos de pagos).



In [4]:
# Diagrama de Clases UML

from plantuml import PlantUML
from IPython.display import Image

plantuml = PlantUML(url='http://www.plantuml.com/plantuml/png/')

uml_code = r"""
@startuml

' Skinparam configurations for better visualization
skinparam class {
  BackgroundColor White
  BorderColor Black
  ArrowColor #444444
  FontName Helvetica
}
skinparam classFontStyle bold
skinparam stereotypeCBackgroundColor #DDDDDD

' Abstract classes (marked with {abstract} or abstract keyword)
abstract class Usuario {
  {abstract} -id : String
  {abstract} -contraseña : String
  {abstract} -rol : String
  {abstract} +autenticar() : Boolean
}

' Final classes (marked with <<final>> stereotype)
class Asignatura <<final>> {
  -código : String
  -nombre : String
  -horario : String
  -docente : String
  -cupos : int
  +verificarCupo() : Boolean
}

class Periodo <<final>> {
  -fecha_inicio : Date
  -fecha_cierre : Date
  -tipo : String
}

class Reporte <<final>> {
  -criterios : Map<String, String>
  -formato : String
  +exportar() : void
}

' Concrete classes
class Estudiante {
  -carrera : String
  -sede : String
  +consultarOferta() : List<Asignatura>
  +matricular() : Matricula
}

class Administrativo {
  -permisos : List<String>
  +gestionarPeriodo(inicio: Date, fin: Date) : void
  +generarReporte(criterios: Map) : Reporte
}

class JefeCarrera {
  -carrera_asignada : String
  +asignarCupos(asignatura: Asignatura, cupos: int) : void
}

class Matricula {
  -fecha : Date
  -estado : String
  +agregarAsignatura(asignatura: Asignatura) : void
  +confirmar() : Boolean
}

class OfertaAcademica {
  -ciclo : String
  -carrera : String
  -sede : String
  +obtenerAsignaturas() : List<Asignatura>
}

' Relationships with proper syntax
Usuario <|-- Estudiante
Usuario <|-- Administrativo
Usuario <|-- JefeCarrera

OfertaAcademica "1" *-- "0..*" Asignatura : contiene >
Matricula "1" o-- "1..*" Asignatura : incluye >
Estudiante "1" --> "0..*" Matricula : realiza >
JefeCarrera "1" --> "0..*" Asignatura : asigna cupos >
Administrativo "1" --> "0..*" Periodo : gestiona >
Administrativo "1" --> "0..*" Reporte : genera >

' Association class example
Estudiante "1" -- "0..*" OfertaAcademica
(Estudiante, OfertaAcademica) .. Consulta

class Consulta {
  -fecha: Date
  +registrar()
}

' Notes and annotations
note top of Usuario : Clase abstracta que representa\na todos los usuarios del sistema

note right of Matricula : Puede estar en estados:\n- Pendiente\n- Confirmada\n- Cancelada

note as N1
  <b>Importante:</b>
  Las relaciones de agregación (*--) indican que
  las asignaturas pueden existir
  independientemente de la oferta académica.
end note
OfertaAcademica .. N1

@enduml
"""

try:
    # Generar la URL de la imagen
    image_url = plantuml.get_url(uml_code)
    # Mostrar la imagen en el notebook
    display(Image(url=image_url))
except Exception as e:
    print(f"Error al generar el diagrama: {str(e)}")
